# UD2 · Práctica · Selenium

**MP5074 Sistemas de Big Data**

En esta práctica extraes datos de **páginas dinámicas** con Selenium usando la web de prácticas
**quotes.toscrape.com**, que tiene varias versiones de la misma página de citas:

| Versión | Qué la hace especial |
|---|---|
| `/` | Estática: las citas vienen en el HTML |
| `/js/` | Las citas las pinta un JavaScript |
| `/js-delayed/` | Igual, pero el JavaScript tarda unos segundos |
| `/login` | Formulario de acceso (acepta cualquier usuario y contraseña) |
| `/scroll` | Scroll infinito: las citas se cargan al bajar |

Todo lo necesario está en el [manual de Selenium](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/).

**Cómo trabajar**

- Resuelve cada celda **Tu turno** antes de seguir. Debajo de cada ejercicio hay un bloque **✔ Comprueba** con lo que debes obtener.
- Cierra siempre el navegador (`try/finally` + `driver.quit()`).
- Mientras desarrollas, abre el navegador **con ventana** (`abrir_navegador(headless=False)`) para ver lo que hace.
- Antes de entregar: *Kernel → Restart Kernel and Run All Cells*.

> Necesitas el entorno `bigdata` con `selenium`, `requests`, `beautifulsoup4`, `pandas` y `pyarrow`, y Google Chrome instalado.

## 0. Preparación

Ejecuta esta celda sin cambiarla. `abrir_navegador` crea el navegador y su espera explícita (manual, sección 11.3).

In [ ]:
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE = "https://quotes.toscrape.com"     # web de prácticas
SALIDA = Path("salida")                  # aquí guardaremos los resultados
SALIDA.mkdir(exist_ok=True)


def abrir_navegador(headless=True, timeout=10):
    """Devuelve (driver, espera). headless=False para ver la ventana del navegador."""
    opciones = webdriver.ChromeOptions()
    if headless:
        opciones.add_argument("--headless=new")
    opciones.add_argument("--window-size=1366,900")
    driver = webdriver.Chrome(options=opciones)
    return driver, WebDriverWait(driver, timeout)


print("selenium listo")

## 1. ¿Estática o dinámica?

Manual: [sección 1](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#1-páginas-estáticas-y-dinámicas).

> ### Tu turno · Ejercicio 1
>
> Con `requests` y BeautifulSoup (sin Selenium):
>
> 1. Descarga `BASE + "/"` y cuenta los elementos `div.quote` → guárdalo en `n_estatica`.
> 2. Descarga `BASE + "/js/"` y cuenta los `div.quote` → `n_js`.
> 3. Comprueba si el texto `"var data"` aparece en el HTML de `/js/` → `tiene_script` (`True`/`False`).
>
> ¿Por qué `n_js` sale distinto de `n_estatica` si en el navegador las dos páginas se ven iguales?
>
> **✔ Comprueba:** `n_estatica = 10`, `n_js = 0`, `tiene_script = True`.

In [ ]:
# Tu código aquí
n_estatica = ...
n_js = ...
tiene_script = ...
print(n_estatica, n_js, tiene_script)

**Tu respuesta:** …

## 2. Primer navegador

Manual: [sección 4](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#4-instalación-y-primer-navegador) y [sección 7](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#7-esperas-la-clave-de-la-estabilidad).

> ### Tu turno · Ejercicio 2
>
> Abre el navegador, ve a `BASE + "/js/"` y, con una **espera explícita**, espera a que haya elementos `div.quote`. Después:
>
> 1. Muestra el título de la pestaña (`driver.title`).
> 2. Guarda en `n_citas` cuántas citas hay.
> 3. Guarda en `primera_autor` el autor de la primera cita (`small.author`).
>
> Usa `try/finally` para cerrar el navegador.
>
> **✔ Comprueba:** `n_citas = 10` y `primera_autor` es un nombre (no está vacío).

In [ ]:
driver, espera = abrir_navegador()
try:
    # Tu código aquí
    pass
finally:
    driver.quit()

print(n_citas, primera_autor)

## 3. Extraer una página con Selenium

Manual: [sección 5](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#5-localizar-elementos-by).

> ### Tu turno · Ejercicio 3
>
> Completa la función `extraer_pagina(driver)`: debe devolver una **lista de diccionarios**, uno por cita de la página
> actual, con las claves `texto` (`span.text`), `autor` (`small.author`) y `etiquetas` (lista con el texto de cada `a.tag`).
> Después pruébala con la primera página de `/js/`.
>
> **✔ Comprueba:** 10 diccionarios; cada uno con las claves `texto`, `autor` y `etiquetas`; `etiquetas` es una lista.

In [ ]:
def extraer_pagina(driver):
    """Devuelve una lista de diccionarios (texto, autor, etiquetas) con las citas de la página actual."""
    filas = []
    # Tu código aquí
    return filas


driver, espera = abrir_navegador()
try:
    driver.get(BASE + "/js/")
    espera.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.quote")))
    pagina1 = extraer_pagina(driver)
finally:
    driver.quit()

print(len(pagina1))
pagina1[0]

## 4. Todas las páginas con clics

Manual: [sección 8](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#8-recorrer-varias-páginas).

> ### Tu turno · Ejercicio 4
>
> Empezando en `BASE + "/js/"`, recorre **todas** las páginas pulsando el botón **Next** (`li.next a`) y guarda todas las
> citas en la lista `citas_sel` usando `extraer_pagina`.
>
> - Después de cada clic, espera a que la página anterior desaparezca (`EC.staleness_of`).
> - Termina cuando no haya botón Next (`find_elements` devuelve una lista vacía).
> - Cuenta las páginas recorridas en `n_paginas` y mide el tiempo total en `t_selenium` (`time.perf_counter()`).
>
> **✔ Comprueba:** `n_paginas = 10`, `len(citas_sel) = 100` y ningún texto repetido.

In [ ]:
citas_sel = []
n_paginas = 0
t0 = time.perf_counter()

driver, espera = abrir_navegador()
try:
    driver.get(BASE + "/js/")
    # Tu código aquí
finally:
    driver.quit()

t_selenium = time.perf_counter() - t0
print(n_paginas, len(citas_sel), f"{t_selenium:.1f} s")
print("Repetidas:", len(citas_sel) - len({c["texto"] for c in citas_sel}))

## 5. Esperas

Manual: [sección 7](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#7-esperas-la-clave-de-la-estabilidad).

En `/js-delayed/` las citas tardan **unos segundos** en aparecer.

> ### Tu turno · Ejercicio 5
>
> En un solo navegador:
>
> 1. Ve a `BASE + "/js-delayed/"` y, **sin esperar**, cuenta los `div.quote` con `find_elements` → `sin_esperar`.
> 2. Después espera con `WebDriverWait` (como máximo **20 s**) a que haya citas → `con_espera`, y mide cuánto ha tardado
>    la espera en `t_espera`.
>
> Responde: ¿qué pasaría si en lugar de la espera explícita pusieras `time.sleep(2)`? ¿Y `time.sleep(30)`?
>
> **✔ Comprueba:** `sin_esperar = 0`, `con_espera = 10` y `t_espera` es de unos segundos (no 20).

In [ ]:
driver, espera = abrir_navegador()
try:
    # Tu código aquí
    pass
finally:
    driver.quit()

print(sin_esperar, con_espera, f"{t_espera:.1f} s")

**Tu respuesta:** …

## 6. Formulario de acceso

Manual: [sección 9](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#9-formularios).

> ### Tu turno · Ejercicio 6
>
> Ve a `BASE + "/login"`, escribe un usuario y una contraseña **inventados** en los campos `#username` y `#password`,
> pulsa el botón de enviar (`input[type='submit']`) y espera a que aparezca el enlace **Logout**
> (`By.LINK_TEXT`). Guarda en `url_final` la URL a la que llegas.
>
> **✔ Comprueba:** la espera no da `TimeoutException` y `url_final` es la portada de la web.

In [ ]:
driver, espera = abrir_navegador()
try:
    # Tu código aquí
    pass
finally:
    driver.quit()

print(url_final)

## 7. Scroll infinito

Manual: [sección 10](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#10-scroll-infinito).

> ### Tu turno · Ejercicio 7
>
> En `BASE + "/scroll"` no hay botón Next: las citas se cargan al bajar. Baja hasta el final de la página una y otra vez
> hasta que el número de `div.quote` **deje de aumentar**, y guarda el total en `n_scroll`.
>
> Pista: `driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")` y una espera con
> `lambda d: len(d.find_elements(...)) > antes`.
>
> **✔ Comprueba:** `n_scroll = 100`.

In [ ]:
driver, espera = abrir_navegador()
try:
    driver.get(BASE + "/scroll")
    # Tu código aquí
finally:
    driver.quit()

print(n_scroll)

## 8. Selenium + BeautifulSoup

Manual: [sección 11](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#11-selenium--beautifulsoup).

> ### Tu turno · Ejercicio 8
>
> 1. Escribe `extraer(html)`: lo mismo que `extraer_pagina`, pero con **BeautifulSoup** sobre un HTML (sin Selenium).
> 2. Recorre las 10 páginas yendo directamente a `BASE + f"/js/page/{n}/"` con `driver.get`. En cada una, espera a las
>    citas, haz la foto con `driver.page_source` y extrae con `extraer`. Guarda todo en `citas_bs` y mide el tiempo en `t_bs`.
>
> Compara `t_bs` con `t_selenium` del ejercicio 4. ¿Cuál es más rápido y por qué?
>
> **✔ Comprueba:** `len(citas_bs) = 100` y los autores salen en el mismo orden que en el ejercicio 4:
> `[c["autor"] for c in citas_bs] == [c["autor"] for c in citas_sel]` es `True`.

In [ ]:
def extraer(html):
    """BeautifulSoup: de un HTML a una lista de diccionarios (texto, autor, etiquetas)."""
    # Tu código aquí
    return []


citas_bs = []
t0 = time.perf_counter()
driver, espera = abrir_navegador()
try:
    # Tu código aquí
    pass
finally:
    driver.quit()
t_bs = time.perf_counter() - t0

mismos = [c["autor"] for c in citas_bs] == [c["autor"] for c in citas_sel]
print(len(citas_bs), mismos, f"BeautifulSoup: {t_bs:.1f} s · solo Selenium: {t_selenium:.1f} s")

**Tu respuesta:** …

## 9. Sin navegador: la API de la página

Manual: [sección 2](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#2-antes-de-selenium-la-pestaña-red).

Abre `quotes.toscrape.com/scroll` en tu navegador con las herramientas de desarrollador (**F12 → Red → Fetch/XHR**) y baja
por la página: verás peticiones a `/api/quotes?page=1`, `?page=2`… que devuelven JSON.

> ### Tu turno · Ejercicio 9
>
> Con `requests`, pide `BASE + "/api/quotes"` con `params={"page": n}` empezando en 1 y sigue mientras `has_next`
> sea `True`. Junta todas las citas en `citas_api` (lista de diccionarios tal como vienen) y cuenta las peticiones en
> `n_peticiones`. Deja una pausa de 0,5 s entre peticiones.
>
> **✔ Comprueba:** `n_peticiones = 10` y `len(citas_api) = 100`.

In [ ]:
citas_api = []
n_peticiones = 0
# Tu código aquí

print(n_peticiones, len(citas_api))

## 10. Análisis con pandas y guardar

Manual: [sección 12](https://rafafl-wirtz.github.io/mp5074-sistemas-big-data/ud2/manuales/selenium/manual_selenium/#12-de-los-datos-a-pandas).

> ### Tu turno · Ejercicio 10
>
> 1. Crea el DataFrame `citas_df` a partir de `citas_bs`.
> 2. Muestra los 5 autores con más citas.
> 3. Con `explode`, muestra las 10 etiquetas más usadas.
> 4. Crea `api_df` con `pd.json_normalize(citas_api)` y comprueba que los textos de las dos fuentes son los mismos:
>    guarda en `diferencias` cuántos textos están en una fuente y no en la otra (pista: `merge` con `how="outer"` e
>    `indicator=True` sobre la columna de texto, o conjuntos de Python).
> 5. Guarda `citas_df` en `salida/citas.parquet` y en `salida/citas.csv`.
>
> **✔ Comprueba:** `citas_df` tiene 100 filas y 3 columnas; `diferencias = 0`; existen los dos ficheros.

In [ ]:
# Tu código aquí
citas_df = ...
diferencias = ...

print(citas_df.shape, diferencias)

## 11. Para terminar

Responde brevemente:

1. ¿Qué tres señales te indican que una página es dinámica?
2. En esta práctica has obtenido las mismas 100 citas de cuatro formas (clics, `driver.get` + BeautifulSoup, scroll y API).
   ¿Cuál usarías si tuvieras que hacerlo cada día y por qué?
3. ¿Por qué es imprescindible `driver.quit()` y por qué se pone dentro de `finally`?

**Tus respuestas:** …